# Lesson 3: API Design

---
## 1. Setup

In [ ]:
import httpx
import json
import time
import threading

BASE = 'http://localhost:8000/api/v1'

def show(resp):
    """Pretty-print a response: status code + formatted JSON body."""
    print(f'HTTP {resp.status_code}')
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

### Verify the OpenSearch index is populated

The API reads from the `machine_usage` index. Make sure it has data before starting the server.
If the count below is not 814, re-run the Lesson 2 notebook to reload the index.

In [ ]:
from opensearchpy import OpenSearch

_os = OpenSearch(hosts=[{'host': 'localhost', 'port': 9200}], use_ssl=False)

info = _os.info()
print(f"OpenSearch {info['version']['number']} ✓")

count = _os.count(index='machine_usage')['count']
print(f'machine_usage index: {count} documents')

assert count > 0, 'Index is empty — re-run the Lesson 2 notebook to load data.'
print('Index ready ✓')

### Step 1: Data Models

Models define the shape of every API response — the contract between the server and any consumer.
We use Pydantic so FastAPI can validate and serialize responses automatically.

In [ ]:
from pydantic import BaseModel
from typing import Optional


class MachineAggregationBucket(BaseModel):
    machine_id: str
    total_duration_hours: float
    average_duration_hours: float
    record_count: int


class PersonnelAggregationBucket(BaseModel):
    personnel_id: str
    total_duration_hours: float
    average_duration_hours: float
    record_count: int


class AggregationResponse(BaseModel):
    total_matching: int
    total_groups: int
    returned: int
    omitted: int


class MachineAggregationResponse(AggregationResponse):
    results: list[MachineAggregationBucket]


class PersonnelAggregationResponse(AggregationResponse):
    results: list[PersonnelAggregationBucket]


class ShiftBucket(BaseModel):
    shift: str
    hours: str
    total_idle_hours: float
    maintenance_idle_hours: float
    operator_gap_idle_hours: float
    maintenance_events: int
    utilization_pct: float


class ShiftComparisonResponse(BaseModel):
    shifts: list[ShiftBucket]


class MaintenanceIssueBucket(BaseModel):
    machine_id: str
    reason_code: str
    total_duration_hours: float
    average_duration_hours: float
    record_count: int


class MaintenanceIssuesResponse(BaseModel):
    total_matching_records: int
    returned: int
    results: list[MaintenanceIssueBucket]


class RecordItem(BaseModel):
    id: str
    machine_id: str
    personnel_id: str
    title: str
    type: str
    start_timestamp: str
    end_timestamp: str
    duration_hours: float
    shift_name: str
    reason_code: Optional[str] = None
    employee_idle_hours_from_prev: Optional[float] = None
    machine_idle_hours_from_prev: Optional[float] = None


class RecordsResponse(BaseModel):
    total_matching: int
    returned: int
    records: list[RecordItem]


print('Models defined ✓')


In [ ]:
# Models for the three endpoints introduced in Section 7.
# Defined here so they are available when the route handlers are registered below.

class PersonnelIssueBucket(BaseModel):
    machine_id: str
    reason_code: str
    total_duration_hours: float
    average_duration_hours: float
    record_count: int


class MaintenancePersonnelBucket(BaseModel):
    personnel_id: str
    personnel_name: Optional[str] = None
    total_duration_hours: float
    average_duration_hours: float
    record_count: int
    issues: list[PersonnelIssueBucket]


class PersonnelLookupRequest(BaseModel):
    issues: list[dict]
    start_date: str | None = None
    end_date: str | None = None
    limit: int = 10


class MaintenancePersonnelResponse(BaseModel):
    returned: int
    results: list[MaintenancePersonnelBucket]


class MaintenanceDoc(BaseModel):
    reason_code: str
    reason_type: str
    document: str


class MaintenanceDocsRequest(BaseModel):
    reason_codes: list[str]
    limit: int = 5


class MaintenanceDocsResponse(BaseModel):
    docs: list[MaintenanceDoc]


class RetrainingRequest(BaseModel):
    machine_id: str
    personnel_id: str
    reason: str
    recommendation_basis: str | None = None


class RetrainingResponse(BaseModel):
    status: str
    message: str
    record_id: int | None = None
    machine_id: str | None = None
    personnel_id: str | None = None
    reason: str | None = None


print('Extended models defined ✓')

### Step 2: OpenSearch Query Functions

Each function translates an API request into an OpenSearch query and returns a typed response object.
The shared `_build_query` helper constructs the filter clauses; the endpoint-specific functions
add the aggregation or hits logic on top.

All six aggregation endpoints share the same `aggregate()` function — they differ only in which
field to group by, which record type to filter on, and which gap field to use for `mode=idle`.

In [ ]:
INDEX = 'machine_usage'

VALID_SHIFTS = {'day': 'Day', 'swing': 'Swing', 'night': 'Night'}

_SORT_KEY_TO_AGG = {
    'total_duration_hours': 'total_duration',
    'average_duration_hours': 'avg_duration',
    'record_count': '_count',
}

_SHIFT_HOURS = {'Day': '06:00-14:00', 'Swing': '14:00-22:00', 'Night': '22:00-06:00'}


def _build_query(*, record_type=None, shift=None, start_date=None, end_date=None,
                 gap_field=None, machine_id=None, personnel_id=None, reason_code=None):
    filters = []
    if record_type:
        filters.append({'term': {'type': record_type}})
    if shift:
        filters.append({'term': {'shift_name': VALID_SHIFTS[shift]}})
    if start_date or end_date:
        r = {}
        if start_date:
            r['gte'] = start_date
        if end_date:
            r['lte'] = end_date
        filters.append({'range': {'start_timestamp': r}})
    if gap_field:
        filters.append({'range': {gap_field: {'gt': 0.25}}})
    if machine_id:
        filters.append({'term': {'machine_id': machine_id}})
    if personnel_id:
        filters.append({'term': {'personnel': personnel_id}})
    if reason_code:
        filters.append({'term': {'reason_code': reason_code}})
    return {'match_all': {}} if not filters else {'bool': {'filter': filters}}


def aggregate(os_client, *, group_by, record_type=None, shift=None, start_date=None,
              end_date=None, limit=10, sort_key='total_duration_hours', mode='active',
              gap_field=None, machine_id=None, personnel_id=None, reason_code=None):
    metric_field = gap_field if mode == 'idle' else 'duration_hours'
    query = _build_query(
        record_type=record_type, shift=shift, start_date=start_date, end_date=end_date,
        gap_field=gap_field if mode == 'idle' else None,
        machine_id=machine_id, personnel_id=personnel_id, reason_code=reason_code,
    )
    body = {
        'size': 0,
        'query': query,
        'aggs': {
            'total_groups': {'cardinality': {'field': group_by}},
            'by_group': {
                'terms': {
                    'field': group_by,
                    'size': limit,
                    'order': {_SORT_KEY_TO_AGG[sort_key]: 'desc'},
                },
                'aggs': {
                    'total_duration': {'sum': {'field': metric_field}},
                    'avg_duration': {'avg': {'field': metric_field}},
                },
            },
        },
    }
    result = os_client.search(index=INDEX, body=body)
    total_matching = result['hits']['total']['value']
    total_groups = result['aggregations']['total_groups']['value']
    buckets = result['aggregations']['by_group']['buckets']
    if group_by == 'machine_id':
        results = [
            MachineAggregationBucket(
                machine_id=b['key'],
                total_duration_hours=round(b['total_duration']['value'], 2),
                average_duration_hours=round(b['avg_duration']['value'], 2),
                record_count=b['doc_count'],
            )
            for b in buckets
        ]
        return MachineAggregationResponse(
            total_matching=total_matching,
            total_groups=total_groups,
            returned=len(results),
            omitted=total_groups - len(results),
            results=results,
        )
    else:
        results = [
            PersonnelAggregationBucket(
                personnel_id=b['key'],
                total_duration_hours=round(b['total_duration']['value'], 2),
                average_duration_hours=round(b['avg_duration']['value'], 2),
                record_count=b['doc_count'],
            )
            for b in buckets
        ]
        return PersonnelAggregationResponse(
            total_matching=total_matching,
            total_groups=total_groups,
            returned=len(results),
            omitted=total_groups - len(results),
            results=results,
        )


def shift_comparison_query(os_client, *, start_date=None, end_date=None):
    query = _build_query(start_date=start_date, end_date=end_date)
    body = {
        'size': 0,
        'query': query,
        'aggs': {
            'by_shift': {
                'terms': {'field': 'shift_name', 'size': 3},
                'aggs': {
                    'total_duration': {'sum': {'field': 'duration_hours'}},
                    'maintenance': {
                        'filter': {'term': {'type': 'Maintenance'}},
                        'aggs': {'duration': {'sum': {'field': 'duration_hours'}}},
                    },
                    'operator_gaps': {
                        'filter': {'range': {'employee_idle_hours_from_prev': {'gt': 0.25}}},
                        'aggs': {'duration': {'sum': {'field': 'employee_idle_hours_from_prev'}}},
                    },
                },
            },
        },
    }
    result = os_client.search(index=INDEX, body=body)
    shifts = []
    for b in result['aggregations']['by_shift']['buckets']:
        maint_idle = round(b['maintenance']['duration']['value'], 1)
        gap_idle = round(b['operator_gaps']['duration']['value'], 1)
        total_idle = round(maint_idle + gap_idle, 1)
        total_dur = b['total_duration']['value']
        utilization = round((1 - total_idle / total_dur) * 100, 1) if total_dur > 0 else 0.0
        shifts.append(ShiftBucket(
            shift=b['key'],
            hours=_SHIFT_HOURS.get(b['key'], ''),
            total_idle_hours=total_idle,
            maintenance_idle_hours=maint_idle,
            operator_gap_idle_hours=gap_idle,
            maintenance_events=b['maintenance']['doc_count'],
            utilization_pct=utilization,
        ))
    return ShiftComparisonResponse(shifts=shifts)


def maintenance_issues_query(os_client, *, shift=None, start_date=None, end_date=None,
                              limit=5, machine_id=None):
    query = _build_query(record_type='Maintenance', shift=shift, start_date=start_date,
                         end_date=end_date, machine_id=machine_id)
    body = {
        'size': 0,
        'query': query,
        'aggs': {
            'by_machine_reason': {
                'multi_terms': {
                    'terms': [{'field': 'machine_id'}, {'field': 'reason_code'}],
                    'size': limit,
                    'order': {'total_duration': 'desc'},
                },
                'aggs': {
                    'total_duration': {'sum': {'field': 'duration_hours'}},
                    'avg_duration': {'avg': {'field': 'duration_hours'}},
                },
            },
        },
    }
    result = os_client.search(index=INDEX, body=body)
    results = [
        MaintenanceIssueBucket(
            machine_id=b['key'][0],
            reason_code=b['key'][1],
            total_duration_hours=round(b['total_duration']['value'], 2),
            average_duration_hours=round(b['avg_duration']['value'], 2),
            record_count=b['doc_count'],
        )
        for b in result['aggregations']['by_machine_reason']['buckets']
    ]
    return MaintenanceIssuesResponse(
        total_matching_records=result['hits']['total']['value'],
        returned=len(results),
        results=results,
    )


def search_records(os_client, *, machine_id=None, personnel_id=None, record_type=None,
                   shift=None, start_date=None, end_date=None, limit=20, sort='asc'):
    query = _build_query(record_type=record_type, shift=shift, start_date=start_date,
                         end_date=end_date, machine_id=machine_id, personnel_id=personnel_id)
    body = {'size': limit, 'query': query, 'sort': [{'start_timestamp': {'order': sort}}]}
    result = os_client.search(index=INDEX, body=body)
    hits = result['hits']['hits']
    records = [
        RecordItem(
            id=h['_source'].get('id', h['_id']),
            machine_id=h['_source']['machine_id'],
            personnel_id=h['_source']['personnel'],
            title=h['_source']['title'],
            type=h['_source']['type'],
            start_timestamp=h['_source']['start_timestamp'],
            end_timestamp=h['_source']['end_timestamp'],
            duration_hours=h['_source']['duration_hours'],
            shift_name=h['_source']['shift_name'],
            reason_code=h['_source'].get('reason_code'),
            employee_idle_hours_from_prev=h['_source'].get('employee_idle_hours_from_prev'),
            machine_idle_hours_from_prev=h['_source'].get('machine_gap_hours_from_prev'),
        )
        for h in hits
    ]
    return RecordsResponse(
        total_matching=result['hits']['total']['value'],
        returned=len(records),
        records=records,
    )


print('Query functions defined ✓')


### Step 3: Define the API

The FastAPI app declares every endpoint: its path, parameters, response model, and the query function
that backs it. This is the layer that would be generated from `docs/lesson_3_api.yaml` in a
Swagger-first workflow.

Once the server starts, Swagger UI will be available at **http://localhost:8000/docs**.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from enum import Enum
from datetime import date


def get_client():
    yield OpenSearch(hosts=[{'host': 'localhost', 'port': 9200}], use_ssl=False)


class Shift(str, Enum):
    day = 'day'
    swing = 'swing'
    night = 'night'


class SortKey(str, Enum):
    total_duration_hours = 'total_duration_hours'
    average_duration_hours = 'average_duration_hours'
    record_count = 'record_count'


class Mode(str, Enum):
    active = 'active'
    idle = 'idle'


class RecordSort(str, Enum):
    asc = 'asc'
    desc = 'desc'


def _common_params(
    shift: Shift | None = Query(None),
    limit: int = Query(10, ge=1, le=100),
    sort_key: SortKey = Query(SortKey.total_duration_hours),
    start_date: date | None = Query(None),
    end_date: date | None = Query(None),
    machine_id: str | None = Query(None),
    personnel_id: str | None = Query(None),
) -> dict:
    return {
        'shift': shift.value if shift else None,
        'limit': limit,
        'sort_key': sort_key.value,
        'start_date': start_date.isoformat() if start_date else None,
        'end_date': end_date.isoformat() if end_date else None,
        'machine_id': machine_id,
        'personnel_id': personnel_id,
    }


def _common_params_with_mode(
    mode: Mode = Query(Mode.active),
    params: dict = Depends(_common_params),
) -> dict:
    return {**params, 'mode': mode.value}


app = FastAPI(
    title='Machine Usage Analytics API — Lesson 3',
    description='Analytics API for querying machine usage data from a manufacturing floor.',
    version='1.0.0',
)
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])


# ── Untyped aggregation views ────────────────────────────────────────────

@app.get('/api/v1/personnel', response_model=PersonnelAggregationResponse, tags=['Personnel'],
         summary='Duration by personnel – all record types')
def get_personnel(params: dict = Depends(_common_params), os=Depends(get_client)):
    return aggregate(os, group_by='personnel', mode='active', **params)


@app.get('/api/v1/machines', response_model=MachineAggregationResponse, tags=['Machines'],
         summary='Duration by machine – all record types')
def get_machines(params: dict = Depends(_common_params_with_mode), os=Depends(get_client)):
    return aggregate(os, group_by='machine_id', gap_field='machine_gap_hours_from_prev', **params)


# ── Regular-only views ────────────────────────────────────────────────────

@app.get('/api/v1/regular/personnel', response_model=PersonnelAggregationResponse, tags=['Personnel'],
         summary='Duration by personnel – Regular sessions only')
def get_regular_personnel(params: dict = Depends(_common_params_with_mode), os=Depends(get_client)):
    return aggregate(os, group_by='personnel', record_type='Regular',
                     gap_field='employee_idle_hours_from_prev', **params)


@app.get('/api/v1/regular/machines', response_model=MachineAggregationResponse, tags=['Machines'],
         summary='Duration by machine – Regular sessions only')
def get_regular_machines(params: dict = Depends(_common_params_with_mode), os=Depends(get_client)):
    return aggregate(os, group_by='machine_id', record_type='Regular',
                     gap_field='machine_gap_hours_from_prev', **params)


# ── Maintenance-only views ────────────────────────────────────────────────

@app.get('/api/v1/maintenance/personnel', response_model=PersonnelAggregationResponse, tags=['Personnel'],
         summary='Duration by personnel – Maintenance events only')
def get_maintenance_personnel(
    params: dict = Depends(_common_params),
    reason_code: str | None = Query(None, description='Filter to a specific failure reason code UUID'),
    os=Depends(get_client),
):
    return aggregate(os, group_by='personnel', record_type='Maintenance', mode='active',
                     reason_code=reason_code, **params)


@app.get('/api/v1/maintenance/machines', response_model=MachineAggregationResponse, tags=['Machines'],
         summary='Duration by machine – Maintenance events only')
def get_maintenance_machines(
    params: dict = Depends(_common_params_with_mode),
    reason_code: str | None = Query(None, description='Filter to a specific failure reason code UUID'),
    os=Depends(get_client),
):
    return aggregate(os, group_by='machine_id', record_type='Maintenance',
                     gap_field='machine_gap_hours_from_prev', reason_code=reason_code, **params)


# ── Shift comparison ──────────────────────────────────────────────────────

@app.get('/api/v1/shifts/comparison', response_model=ShiftComparisonResponse, tags=['Shifts'],
         summary='Compare idle time and utilization across shifts')
def get_shift_comparison(
    start_date: date | None = Query(None),
    end_date: date | None = Query(None),
    os=Depends(get_client),
):
    return shift_comparison_query(
        os,
        start_date=start_date.isoformat() if start_date else None,
        end_date=end_date.isoformat() if end_date else None,
    )


# ── Maintenance issues ────────────────────────────────────────────────────

@app.get('/api/v1/maintenance/issues', response_model=MaintenanceIssuesResponse,
         tags=['Maintenance'], summary='Top maintenance issues by downtime (machine + reason code)')
def get_maintenance_issues(
    shift: Shift | None = Query(None),
    limit: int = Query(5, ge=1, le=100),
    machine_id: str | None = Query(None),
    start_date: date | None = Query(None),
    end_date: date | None = Query(None),
    os=Depends(get_client),
):
    return maintenance_issues_query(
        os,
        shift=shift.value if shift else None,
        machine_id=machine_id,
        start_date=start_date.isoformat() if start_date else None,
        end_date=end_date.isoformat() if end_date else None,
        limit=limit,
    )


# ── Raw record search ─────────────────────────────────────────────────────

@app.get('/api/v1/records', response_model=RecordsResponse, tags=['Records'],
         summary='Search individual records with optional filters')
def get_records(
    machine_id: str | None = Query(None),
    personnel_id: str | None = Query(None),
    type: str | None = Query(None),
    shift: Shift | None = Query(None),
    limit: int = Query(20, ge=1, le=100),
    sort: RecordSort = Query(RecordSort.asc),
    start_date: date | None = Query(None),
    end_date: date | None = Query(None),
    os=Depends(get_client),
):
    return search_records(
        os,
        machine_id=machine_id,
        personnel_id=personnel_id,
        record_type=type.capitalize() if type else None,
        shift=shift.value if shift else None,
        start_date=start_date.isoformat() if start_date else None,
        end_date=end_date.isoformat() if end_date else None,
        limit=limit,
        sort=sort.value,
    )


print('API defined ✓  —  9 core endpoints registered')
print('API defined ✓  —  9 core endpoints registered')
print('Section 7 adds the remaining 3 endpoints and restarts the server.')


### Start the Server

9 core endpoints are defined — uvicorn starts with the base route table. Section 7 will add the remaining 3 endpoints and restart the server with the full set.

The cell after this one waits until the server is responding before continuing.


In [ ]:
import uvicorn

_uvicorn_server = None

def _start_api_server():
    global _uvicorn_server
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
    _uvicorn_server = uvicorn.Server(config)
    threading.Thread(target=_uvicorn_server.run, daemon=True).start()

_start_api_server()
print('API server starting on port 8000...')
print('Run the next cell to confirm it is ready.')


In [ ]:
print('Waiting for API...', end='', flush=True)
for attempt in range(30):
    try:
        resp = httpx.get(f'{BASE}/machines', params={'limit': 1})
        resp.raise_for_status()
        print(f' up! (attempt {attempt + 1})')
        print(f'API is ready ✓  →  http://localhost:8000/docs')
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print('\nAPI did not respond after 60 seconds.')
    print('Check for errors above and try restarting the kernel.')

---
## 2. The Mission and the Contract

**Goal:** Understand *why* the API is shaped the way it is, before making a single call.

---

### The Agent's Mission

The system we're building is an AI agent that analyzes machine usage and maintenance data to answer one question:

> *Which machines are underperforming, why are they breaking down, and which personnel may need retraining?*

The API is the agent's only window into the data. Every endpoint exists because the agent needs to answer a specific question in that investigation.

---

### The Analyst's Checklist → API Shape

Before writing any code, we asked: *if a factory analyst did this job manually, what questions would they ask?* Each question maps to one API call:

| Question the analyst asks | Endpoint that answers it |
|---------------------------|-------------------------|
| Which machines have the most maintenance downtime? | `GET /maintenance/machines` |
| For a specific machine, what failure types are causing the most downtime? | `GET /maintenance/issues?machine_id=...` |
| Who are the maintenance technicians on that machine? | `GET /maintenance/personnel?machine_id=...` |
| Which operators use that machine most? | `GET /regular/personnel?machine_id=...` |
| How do problems compare across Day, Swing, and Night shifts? | `GET /shifts/comparison` |
| What *exactly* happened on that machine — and when? | `GET /records?machine_id=...` |
| What is an operator's full session history across all machines? | `GET /records?personnel_id=...` |

This reveals the key architectural divide:

- **Aggregation endpoints** — answer discovery questions: *which* machines, *who* is involved, *how much* time.
- **The records endpoint** — answers diagnosis questions: *what exactly happened*, in time order.

You need both. Aggregates find the problem. Records explain it.

---

### Swagger as Contract

The Swagger spec at `docs/lesson_3_api.yaml` was written *before* any implementation code. It defines:

- What each endpoint accepts (parameters, request bodies)
- What it returns (response schemas with field names and types)
- What it means (descriptions the agent uses to choose the right tool)

Writing the spec first forces you to think through the interface without getting distracted by implementation. The code generator (or a colleague) then has an unambiguous target to build against.

In Lesson 4, we'll wrap these endpoints as MCP tools. The tool descriptions come directly from the Swagger — which is why those descriptions need to be precise enough for an LLM to reason about.

> **This notebook covers the data layer.** Lesson 4 adds the tooling layer on top.

---
## 3. Discovery — Aggregation Endpoints

**Goal:** Run a live investigation using aggregation endpoints to identify the most problematic machine and trace who is involved.

We'll treat this as a real scenario: *a machine is breaking down frequently. Let's find it.*

Each step captures a UUID from the API response and uses it in the next call, so the investigation threads together from real data.

### Step 3a: Which machine has the most maintenance downtime?

`GET /maintenance/machines` aggregates by machine and sums total maintenance hours. The top result is our starting point.

In [ ]:
resp = httpx.get(f'{BASE}/maintenance/machines', params={'limit': 5})
show(resp)

In [ ]:
data = resp.json()
top_machine = data['results'][0]['machine_id']
top_machine_hours = data['results'][0]['total_duration_hours']
top_machine_count = data['results'][0]['record_count']

print(f'Top machine:     {top_machine}')
print(f'Total maint hrs: {top_machine_hours:.1f}h across {top_machine_count} events')

### Step 3b: What failure types is that machine experiencing?

`GET /maintenance/issues` groups by `(machine_id, reason_code)` — showing which specific failure codes are consuming the most time on this machine.

Notice: `reason_code` is a UUID. It's opaque here — we know *that* a failure type is costly, but not *what* it means. That's an open question we'll come back to.

In [ ]:
resp = httpx.get(f'{BASE}/maintenance/issues', params={'machine_id': top_machine, 'limit': 5})
show(resp)

In [ ]:
issues = resp.json()['results']
top_reason_code = issues[0]['reason_code']

print(f'Top failure code: {top_reason_code}')
print(f'Total hours:      {issues[0]["total_duration_hours"]:.1f}h across {issues[0]["record_count"]} events')
print()
print('All failure codes on this machine:')
for issue in issues:
    print(f'  {issue["reason_code"]}  {issue["total_duration_hours"]:.1f}h  ({issue["record_count"]} events)')

### Step 3c: Who are the maintenance technicians on that machine?

`GET /maintenance/personnel` aggregates by personnel, filtered to our specific machine. These are the people doing the repairs.

In [ ]:
resp = httpx.get(f'{BASE}/maintenance/personnel', params={'machine_id': top_machine, 'limit': 10})
show(resp)

### Step 3d: Which operators use that machine most?

`GET /regular/personnel` aggregates operator sessions only, filtered to this machine. These are the people running it day-to-day. We capture the top operator for use in Section 4.

In [ ]:
resp = httpx.get(f'{BASE}/regular/personnel', params={'machine_id': top_machine, 'limit': 5})
show(resp)

In [ ]:
op_data = resp.json()
top_operator = op_data['results'][0]['personnel_id']
top_operator_hours = op_data['results'][0]['total_duration_hours']

print(f'Top operator: {top_operator}')
print(f'Total hours on this machine: {top_operator_hours:.1f}h')

### Step 3e: How does this compare across shifts?

`GET /shifts/comparison` gives side-by-side utilization metrics across Day, Swing, and Night shifts — useful for spotting structural patterns vs. isolated incidents. A shift with consistently higher maintenance hours may have a staffing or process issue.

In [ ]:
resp = httpx.get(f'{BASE}/shifts/comparison')
show(resp)

In [ ]:
shifts = resp.json()['shifts']

print(f"{'Shift':<8} {'Hours':<14} {'Util%':>6}  {'Maint hrs':>10}  {'Events':>7}")
print('-' * 55)
for s in shifts:
    print(f"{s['shift']:<8} {s['hours']:<14} {s['utilization_pct']:>5.1f}%  {s['maintenance_idle_hours']:>10.1f}h  {s['maintenance_events']:>7}")

---
## 4. Diagnosis — The Records Endpoint

**Goal:** Switch from *discovery* to *diagnosis* — from knowing that a problem exists to seeing exactly what happened.

Aggregates told us **that** a problem exists. Records tell us **what actually happened**.

`GET /records` returns individual (non-aggregated) documents from the index, sorted by timestamp. It's the same data that feeds the aggregations, surfaced one event at a time — so you can see the sequence, the gaps, and the context.

### Step 4a: Full machine timeline

Both Regular operator sessions and Maintenance events, in time order. This is the complete picture of everything that happened on this machine.

In [ ]:
resp = httpx.get(f'{BASE}/records', params={'machine_id': top_machine, 'limit': 20})
show(resp)

In [ ]:
records = resp.json()
print(f"Total sessions on this machine: {records['total_matching']}")
print(f"Returned: {records['returned']}")
print()
print(f"{'Type':<15} {'Start':<22} {'Hrs':>5}  {'Personnel':<12}")
print('-' * 60)
for r in records['records']:
    print(f"{r['type']:<15} {r['start_timestamp'][:19]:<22} {r['duration_hours']:>5.2f}  {r['personnel_id'][:12]}...")

### Step 4b: Maintenance-only view

Filter to `type=maintenance` to isolate just the repair events. Is it the same technician each time? The same `reason_code`? Patterns in the repair history are often the most diagnostic signal.

In [ ]:
resp = httpx.get(f'{BASE}/records', params={'machine_id': top_machine, 'type': 'maintenance', 'limit': 20})
show(resp)

In [ ]:
maint_records = resp.json()
print(f"Total maintenance events on this machine: {maint_records['total_matching']}")
print()
print(f"{'Start':<22} {'Hrs':>5}  {'Reason code':<38}  {'Personnel':<12}")
print('-' * 85)
for r in maint_records['records']:
    reason = r.get('reason_code') or '(none)'
    print(f"{r['start_timestamp'][:19]:<22} {r['duration_hours']:>5.2f}  {reason:<38}  {r['personnel_id'][:12]}...")

### Step 4c: Operator history

Now pivot to the top operator identified in Step 3d. `GET /records?personnel_id=...` returns their full session history across all machines and shifts — useful for spotting whether an operator's issues are machine-specific or follow them across the floor.

In [ ]:
resp = httpx.get(f'{BASE}/records', params={'personnel_id': top_operator, 'limit': 20})
show(resp)

In [ ]:
op_records = resp.json()
print(f"Total sessions for this operator: {op_records['total_matching']}")
print()

machine_counts = {}
for r in op_records['records']:
    machine_counts[r['machine_id']] = machine_counts.get(r['machine_id'], 0) + 1

print('Sessions by machine (first 20 records):')
for mid, cnt in sorted(machine_counts.items(), key=lambda x: -x[1]):
    marker = '  ← top machine from section 3' if mid == top_machine else ''
    print(f'  {mid[:12]}...  {cnt} sessions{marker}')

---
## 5. What's Missing?

We can now find a problematic machine, trace its failure patterns, identify who is involved, and pull the full session timeline. That's a complete analytical picture.

But notice three things we *cannot* do yet:

---

**1. We can see `reason_code: <uuid>` in our results — but what does that UUID actually mean?**

Every maintenance event has a reason code that tells us *why* the machine broke down. Right now it's just a UUID. To understand the failure, we'd need to look up what the code represents — a description, a standard operating procedure, training materials. There's no endpoint for that yet.

---

**2. We found the top `(machine, reason_code)` failure pairs — but drilling into exactly who worked those specific issues requires more than we have.**

We can filter `/maintenance/personnel` by machine. But we can't ask: *"show me everyone who handled this specific failure code on this specific machine."* That requires matching on a composite `(machine_id, reason_code)` pair — a more targeted query than our current endpoints support.

---

**3. We've done the analysis. But we can't act on it.**

If the analysis shows a pattern — say, an operator whose sessions consistently precede the same maintenance failure — there's currently no way to record a retraining recommendation. That action writes to a database, not the index. It's a downstream step that requires human approval before it can be taken.

---

These three gaps are intentional, not oversights. Each one was held back because it requires context we hadn't established yet. Section 7 adds all three.

---
## 6. Summary

| Section | What we did |
|---------|-------------|
| Setup | Verified the OpenSearch index; defined models, query functions, and routes inline; started the server in a background thread |
| Mission & Contract | Mapped the analyst's checklist to API shapes; established the aggregation vs. records divide; introduced Swagger-as-contract |
| Discovery | Used aggregation endpoints to rank machines by maintenance downtime, identify failure codes, find involved personnel, and compare across shifts |
| Diagnosis | Used the records endpoint to trace the machine's full event timeline and the operator's session history across machines |
| What's Missing | Identified three open gaps — reason code lookup, composite personnel lookup, and the retraining write action — filled in Section 7 |
| Completing the API | Added three POST endpoints for composite personnel lookup, documentation retrieval, and retraining submission — all 12 routes now registered |

---
## 7. Three New Endpoints — Explained

All three endpoints are introduced here. The cell below defines them, registers them with the app, and restarts the server so the full route table takes effect.

> **Refresh [http://localhost:8000/docs](http://localhost:8000/docs)** — you should now see all twelve endpoints including the three POST routes added in Section 1.

---

### Why were these three held back?

| Endpoint | Why it waited until now |
|---|---|
| `POST /maintenance/personnel-lookup` | Only meaningful *after* identifying specific `(machine, reason_code)` failure patterns. The composite query concept needed the issues endpoint first. |
| `POST /maintenance/docs` | Reads from **reference data** — a separate source from OpenSearch. Introducing a second data source in a lesson about the index would have been a distraction. |
| `POST /retraining` | The only **write** endpoint. Mixing a high-impact database write into a lesson about analytical queries would have muddied the learning arc. |

In [ ]:
from pathlib import Path
import sqlite3 as _sqlite3

from utils.reference_data import CANNED_DOCS, PERSONNEL_NAMES
print(f'Reference data: {len(PERSONNEL_NAMES)} personnel names, {len(CANNED_DOCS)} reason code entries')


# ── POST /maintenance/personnel-lookup ───────────────────────────────────

def _personnel_lookup_handler(request: PersonnelLookupRequest) -> MaintenancePersonnelResponse:
    should_clauses = [
        {'bool': {'must': [
            {'term': {'machine_id': pair['machine_id']}},
            {'term': {'reason_code': pair['reason_code']}},
        ]}}
        for pair in request.issues
    ]
    filters: list[dict] = [
        {'term': {'type': 'Maintenance'}},
        {'bool': {'should': should_clauses, 'minimum_should_match': 1}},
    ]
    if request.start_date:
        filters.append({'range': {'start_timestamp': {'gte': request.start_date}}})
    if request.end_date:
        filters.append({'range': {'start_timestamp': {'lte': request.end_date}}})
    body = {
        'size': 0,
        'query': {'bool': {'filter': filters}},
        'aggs': {
            'by_personnel': {
                'terms': {'field': 'personnel', 'size': request.limit, 'order': {'total_duration': 'desc'}},
                'aggs': {
                    'total_duration': {'sum': {'field': 'duration_hours'}},
                    'avg_duration': {'avg': {'field': 'duration_hours'}},
                    'by_issue': {
                        'multi_terms': {
                            'terms': [{'field': 'machine_id'}, {'field': 'reason_code'}],
                            'size': 50,
                        },
                        'aggs': {
                            'issue_duration': {'sum': {'field': 'duration_hours'}},
                            'issue_avg': {'avg': {'field': 'duration_hours'}},
                        },
                    },
                },
            },
        },
    }
    result = _os.search(index=INDEX, body=body)
    results = [
        MaintenancePersonnelBucket(
            personnel_id=b['key'],
            personnel_name=PERSONNEL_NAMES.get(b['key']),
            total_duration_hours=round(b['total_duration']['value'], 2),
            average_duration_hours=round(b['avg_duration']['value'], 2),
            record_count=b['doc_count'],
            issues=[
                PersonnelIssueBucket(
                    machine_id=ib['key'][0],
                    reason_code=ib['key'][1],
                    total_duration_hours=round(ib['issue_duration']['value'], 2),
                    average_duration_hours=round(ib['issue_avg']['value'], 2),
                    record_count=ib['doc_count'],
                )
                for ib in b['by_issue']['buckets']
            ],
        )
        for b in result['aggregations']['by_personnel']['buckets']
    ]
    return MaintenancePersonnelResponse(returned=len(results), results=results)


app.add_api_route(
    '/api/v1/maintenance/personnel-lookup', _personnel_lookup_handler,
    methods=['POST'], response_model=MaintenancePersonnelResponse,
    tags=['Maintenance'], summary='Find personnel involved in specific maintenance issues',
)


# ── POST /maintenance/docs ────────────────────────────────────────────────

def _maintenance_docs_handler(request: MaintenanceDocsRequest) -> MaintenanceDocsResponse:
    docs = []
    for code in request.reason_codes:
        entries = CANNED_DOCS.get(code, [])
        docs.extend([
            MaintenanceDoc(reason_code=e.reason_code, reason_type=e.reason_type, document=e.document)
            for e in entries[:request.limit]
        ])
    return MaintenanceDocsResponse(docs=docs)


app.add_api_route(
    '/api/v1/maintenance/docs', _maintenance_docs_handler,
    methods=['POST'], response_model=MaintenanceDocsResponse,
    tags=['Maintenance'], summary='Retrieve documents for maintenance reason codes',
)


# ── POST /retraining ──────────────────────────────────────────────────────

def _retraining_handler(request: RetrainingRequest) -> RetrainingResponse:
    _db_path = Path().resolve().parent / 'data' / 'database.db'
    conn = _sqlite3.connect(str(_db_path))
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            INSERT INTO retraining_requirements
            (machine_id, personnel_id, reason, recommendation_basis, requested_at, requested_by, status)
            VALUES (?, ?, ?, ?, datetime('now'), 'api', 'pending')
            """,
            (request.machine_id, request.personnel_id, request.reason, request.recommendation_basis),
        )
        conn.commit()
        return RetrainingResponse(
            status='success',
            message='Retraining requirement submitted successfully',
            record_id=cursor.lastrowid,
            machine_id=request.machine_id,
            personnel_id=request.personnel_id,
            reason=request.reason,
        )
    except Exception as e:
        conn.rollback()
        raise HTTPException(status_code=500, detail=f'Database error: {e}') from e
    finally:
        conn.close()


app.add_api_route(
    '/api/v1/retraining', _retraining_handler,
    methods=['POST'], response_model=RetrainingResponse,
    tags=['Operations'], summary='Submit a retraining requirement', status_code=201,
)


# ── Restart server to pick up the 3 new routes ────────────────────────────

print('3 endpoints defined ✓  (personnel-lookup, docs, retraining)')
print('Restarting API server to register them...')

if _uvicorn_server:
    _uvicorn_server.should_exit = True
    time.sleep(2)

_start_api_server()
print('Server restarting — run the next cell to confirm it is back up.')


In [ ]:
print('Waiting for server restart...', end='', flush=True)
for attempt in range(15):
    try:
        resp = httpx.get(f'{BASE}/maintenance/personnel-lookup', timeout=1.0)
        # 405 (method not allowed) means the route exists but we sent a GET — that's success
        if resp.status_code in (200, 405, 422):
            print(f' up! (attempt {attempt + 1})')
            print('All 12 endpoints live ✓  →  http://localhost:8000/docs')
            break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    print('\nServer did not respond after restart. Try re-running the cell above.')


### `POST /maintenance/personnel-lookup` — Targeted personnel lookup

`GET /maintenance/personnel?machine_id=...` tells you everyone associated with maintenance on a machine. But there's a more targeted question: *who was present when this specific failure code occurred on this specific machine?*

A note on what "personnel" means here: on a Maintenance record, the personnel field is the employee who **reported** the issue — typically the operator running the machine when it broke (90% of the time), occasionally a shift manager (10%). It is not a repair technician field; no separate technician data is captured. So this endpoint answers: *which operators are repeatedly correlated with a specific failure type?* That is the retraining signal.

The query matches on composite `(machine_id, reason_code)` pairs simultaneously using `bool/should` — not a single field filter. Each result includes a per-issue breakdown so you can see which specific failures each person was present for, not just total hours.

This endpoint is a direct downstream consumer of `GET /maintenance/issues`: the `(machine_id, reason_code)` pairs from that response become the request body here.


In [ ]:
# Demo: find who handled the top (machine, reason_code) pair from Section 3b.
resp = httpx.post(f'{BASE}/maintenance/personnel-lookup', json={
    'issues': [{'machine_id': top_machine, 'reason_code': top_reason_code}],
    'limit': 5,
})
show(resp)

### `POST /maintenance/docs` — Unlock the reason codes

Throughout Sections 3 and 4 we kept seeing `reason_code: <uuid>`. We knew *that* a failure occurred and how many hours it consumed. We couldn't know *what* it was.

This endpoint resolves that. Each reason code maps to up to four document types:

| Document type | What it contains |
|---|---|
| `description` | Technical summary of what this failure mode is |
| `SOP` | Standard operating procedure for handling it |
| `training` | Training materials for operators working with this equipment |
| `support` | Vendor or specialist contact information |

The data lives in `utils/reference_data.py` — a static dictionary, not the OpenSearch index. This is a deliberate architectural choice: reference data that rarely changes doesn't belong in the search index. It's the second data source the agent touches, and it's why this endpoint wasn't in Lesson 3 earlier.

In [ ]:
# Demo: resolve the top reason code from Section 3b to human-readable documentation.
resp = httpx.post(f'{BASE}/maintenance/docs', json={
    'reason_codes': [top_reason_code],
    'limit': 4,
})
show(resp)

### `POST /retraining` — The action endpoint

Every other endpoint in this API reads data. This one writes.

If the analysis confirms a pattern — a technician whose average repair time for a specific failure consistently exceeds the expected range — there needs to be a mechanism to record that finding and queue follow-up action. This endpoint writes a retraining recommendation to the `retraining_requirements` table in SQLite.

The human approval requirement isn't just a convention — it's enforced at the tool layer in Lesson 4. The MCP tool that wraps this endpoint includes an explicit constraint in its description: the agent must present its evidence to a human reviewer and receive explicit confirmation before calling it. That boundary lives in the tool, not the API.

> **Note on the database path:** The handler resolves `data/database.db` relative to the repo root using `Path().resolve().parent`. This works because the notebook runs from the `api/` directory — one level below the repo root where `data/` lives.

In [ ]:
# Demo: submit a retraining request for the top operator on the top machine.
# In practice an agent would only call this after human review.
resp = httpx.post(f'{BASE}/retraining', json={
    'machine_id': top_machine,
    'personnel_id': top_operator,
    'reason': 'Operator sessions consistently precede maintenance events on this machine.',
    'recommendation_basis': f'Identified via lesson 3 investigation: top machine {top_machine[:8]}...',
})
show(resp)

In [ ]:
routes = [
    (', '.join(sorted(r.methods)), r.path)
    for r in app.routes
    if hasattr(r, 'path') and r.path.startswith('/api/v1')
]
routes.sort(key=lambda x: x[1])

print(f'All {len(routes)} endpoints registered:\n')
for methods, path in routes:
    print(f'  {methods:<6}  {path}')
print()
print('Open http://localhost:8000/docs to see the complete Swagger UI ✓')